# PatentRank Phase 3: Fine-Tuning a Cross-Encoder Ranker on Google Colab (T4 GPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)

## 1. System Architecture & Context
In **Phase 1** (BM25 keyword search) and **Phase 2** (Dense vector search via Gemini API + PGVector), candidate retrieval achieved high recall by querying lexical and dense representations independently. However, both bi-encoder dense retrieval and BM25 suffer from a fundamental limitation: **they compute query and document representations independently**.

- **Bi-Encoders (Stage 1 / First-Stage Retrieval):** Encodes Query $E(q)$ and Document $E(d)$ into separate vector representations, scoring with cosine similarity $E(q) \cdot E(d)$. This enables sub-millisecond Approximate Nearest Neighbor (ANN) search across millions of documents, but completely lacks cross-token attention.
- **Cross-Encoders (Stage 2 / Re-Ranking):** Concatenates Query and Document into a single sequence `[CLS] query [SEP] patent_title + patent_abstract [SEP]` and feeds them simultaneously into all transformer self-attention layers. Every token in the query attends to every token in the patent document ($O(L^2)$ attention), allowing the model to capture intricate technical qualifiers, patent claims nuances, and syntactic dependencies.

```
+---------------------------------------------------------------------------------------------------+
|                                   PATENT SEARCH PIPELINE                                          |
|                                                                                                   |
|  User Query: "boundary scan register chain test system"                                           |
|       |                                                                                           |
|       v                                                                                           |
|  [STAGE 1: HIGH-RECALL HYBRID CANDIDATE RETRIEVAL]                                               |
|  +-------------------------------+       +------------------------------------+                   |
|  | BM25 Lexical (Elasticsearch)  |  +    | Dense Vector (Gemini + PGVector)   |                   |
|  +-------------------------------+       +------------------------------------+                   |
|       |                                                    |                                      |
|       +-------------------------> RRF <--------------------+                                      |
|                                    |                                                              |
|                                    v                                                              |
|                    Top 50 Hybrid Candidate Patents                                                |
|                                    |                                                              |
|  [STAGE 2: HIGH-PRECISION CROSS-ENCODER RE-RANKING]                                              |
|  +------------------------------------------------------------------+                             |
|  | Cross-Attention Transformer: [CLS] Query [SEP] Candidate [SEP]   |                             |
|  | Full pairwise token interactions across all transformer layers   |                             |
|  +------------------------------------------------------------------+                             |
|                                    |                                                              |
|                                    v                                                              |
|                    Top 10 High-Precision Patent Results                                           |
+---------------------------------------------------------------------------------------------------+
```

---

## 2. Hard-Negative Mining Methodology

A standard failure mode in ranking model development is training on random negatives. In patent search, a random patent from physics or chemistry is trivial to reject based on superficial keyword overlap.

To build a model that solves the hardest retrieval challenges, we specifically mined **hard negatives** from Stage 1 retrieval:
1. **BM25 Top Misses:** Documents that share high-frequency keyword overlap with the query (e.g., matching "scan", "register", "test"), but represent distinct or orthogonal inventions.
2. **Dense Top Misses:** Documents that are located close to the query in embedding vector space due to topical proximity, but lack the specific inventive mechanism.
3. **Orthogonal Negatives:** Cross-category distractors for contrastive baseline stability.

Our dataset includes **1,120 training pairs** (160 queries $\times$ 1 positive + 6 hard/orthogonal negatives) and **280 validation pairs** (40 queries $\times$ 1 positive + 6 negatives).

In [ ]:
# Step 1: Verify Hardware & GPU Acceleration (Google Colab Free T4)
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name:     {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM:      {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

In [ ]:
# Step 2: Install Required Dependencies
!pip install -q transformers datasets accelerate rank-bm25 tqdm matplotlib pandas

In [ ]:
# Step 3: Load Training and Validation Datasets
import os
import json

# If running directly from cloned repo:
# !git clone https://github.com/alokverma9/patent-search.git && cd patent-search

train_file = "data/train_pairs.jsonl"
val_file = "data/val_pairs.jsonl"

assert os.path.exists(train_file), f"Missing {train_file}. Please ensure repository data files are present."
assert os.path.exists(val_file), f"Missing {val_file}. Please ensure repository data files are present."

with open(train_file, 'r', encoding='utf-8') as f:
    train_samples = [json.loads(line) for line in f if line.strip()]
with open(val_file, 'r', encoding='utf-8') as f:
    val_samples = [json.loads(line) for line in f if line.strip()]

print(f"Loaded {len(train_samples):,} training pairs and {len(val_samples):,} validation pairs.")
print("\nSample Training Pair:")
print(json.dumps(train_samples[0], indent=2))

In [ ]:
# Step 4: Define PyTorch Dataset & Dynamic Collation
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_cosine_schedule_with_warmup

class PatentRankingPairDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        return {
            "query_id": item["query_id"],
            "query": item["query"],
            "doc_id": item["doc_id"],
            "doc_text": item["doc_text"],
            "label": float(item["label"])
        }

def make_collate_fn(tokenizer, max_length=256):
    def collate_fn(batch):
        queries = [item["query"] for item in batch]
        docs = [item["doc_text"] for item in batch]
        labels = [item["label"] for item in batch]
        query_ids = [item["query_id"] for item in batch]
        doc_ids = [item["doc_id"] for item in batch]

        encoded = tokenizer(
            queries,
            docs,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )
        encoded["labels"] = torch.tensor(labels, dtype=torch.float)
        encoded["query_ids"] = query_ids
        encoded["doc_ids"] = doc_ids
        return encoded
    return collate_fn

In [ ]:
# Step 5: Initialize Pretrained Cross-Encoder Model
MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Loading {MODEL_NAME} on {DEVICE}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=1)
model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f"[OK] Loaded Cross-Encoder. Total Parameters: {total_params:,}")

In [ ]:
# Step 6: Evaluation Function (Ranking MRR and Hit@1)
def evaluate_ranking(model, dataloader, device, loss_fn):
    model.eval()
    total_loss = 0.0
    num_batches = 0
    query_scores = {}

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            query_ids = batch["query_ids"]

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits.squeeze(-1)
            loss = loss_fn(logits, labels)

            total_loss += loss.item()
            num_batches += 1

            scores = logits.cpu().tolist()
            lbls = labels.cpu().tolist()
            for q_id, sc, lb in zip(query_ids, scores, lbls):
                query_scores.setdefault(q_id, []).append((sc, lb))

    avg_loss = total_loss / max(1, num_batches)
    mrrs = []
    hit1s = []

    for q_id, candidates in query_scores.items():
        sorted_cands = sorted(candidates, key=lambda x: x[0], reverse=True)
        pos_rank = None
        for r, (sc, lb) in enumerate(sorted_cands, start=1):
            if lb == 1.0:
                pos_rank = r
                break
        if pos_rank is not None:
            mrrs.append(1.0 / pos_rank)
            hit1s.append(1.0 if pos_rank == 1 else 0.0)
        else:
            mrrs.append(0.0)
            hit1s.append(0.0)

    val_mrr = sum(mrrs) / max(1, len(mrrs))
    val_hit1 = sum(hit1s) / max(1, len(hit1s))
    return avg_loss, val_mrr, val_hit1

In [ ]:
# Step 7: Hyperparameters & Pretrained Baseline Evaluation
import torch.nn as nn

BATCH_SIZE = 16
EPOCHS = 3
LR = 2e-5
MAX_LEN = 256
OUTPUT_DIR = "models/patentrank-cross-encoder"

train_loader = DataLoader(PatentRankingPairDataset(train_samples), batch_size=BATCH_SIZE, shuffle=True, collate_fn=make_collate_fn(tokenizer, MAX_LEN))
val_loader = DataLoader(PatentRankingPairDataset(val_samples), batch_size=32, shuffle=False, collate_fn=make_collate_fn(tokenizer, MAX_LEN))
loss_fn = nn.BCEWithLogitsLoss()

# Evaluate Zero-Shot Pretrained Performance
base_loss, base_mrr, base_hit1 = evaluate_ranking(model, val_loader, DEVICE, loss_fn)
print("=" * 60)
print("ZERO-SHOT BASELINE (Pretrained MS MARCO Weights on Patent Data):")
print(f"Validation Loss: {base_loss:.4f}")
print(f"Validation MRR:  {base_mrr:.4f}")
print(f"Validation Hit@1: {base_hit1 * 100:.2f}%")
print("=" * 60)

In [ ]:
# Step 8: Fine-Tuning Execution Loop
import time
from tqdm import tqdm

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * 0.1)
scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)

history = {"epochs": [], "best_val_mrr": base_mrr, "best_epoch": 0}
best_val_mrr = base_mrr

print(f"Starting Fine-Tuning for {EPOCHS} Epochs on {DEVICE}...")
t_start = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}")
    for batch in pbar:
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(outputs.logits.squeeze(-1), labels)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        epoch_loss += loss.item()
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})

    avg_train_loss = epoch_loss / len(train_loader)
    val_loss, val_mrr, val_hit1 = evaluate_ranking(model, val_loader, DEVICE, loss_fn)
    print(f"Epoch {epoch:2d}/{EPOCHS} | Train Loss: {avg_train_loss:.4f} | Val Loss: {val_loss:.4f} | Val MRR: {val_mrr:.4f} | Val Hit@1: {val_hit1*100:.2f}%")

    history["epochs"].append({
        "epoch": epoch,
        "train_loss": avg_train_loss,
        "val_loss": val_loss,
        "val_mrr": val_mrr,
        "val_hit1": val_hit1
    })

    if val_mrr >= best_val_mrr:
        best_val_mrr = val_mrr
        history["best_epoch"] = epoch
        history["best_val_mrr"] = best_val_mrr
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        model.save_pretrained(OUTPUT_DIR)
        tokenizer.save_pretrained(OUTPUT_DIR)
        print(f"  --> Saved new best checkpoint to {OUTPUT_DIR}")

print(f"\nFine-Tuning completed in {time.time() - t_start:.2f} seconds. Best Val MRR: {best_val_mrr:.4f}")

In [ ]:
# Step 9: Plot Training Loss & Ranking MRR Trajectory
import matplotlib.pyplot as plt

epochs_range = [e["epoch"] for e in history["epochs"]]
train_losses = [e["train_loss"] for e in history["epochs"]]
val_losses = [e["val_loss"] for e in history["epochs"]]
val_mrrs = [e["val_mrr"] for e in history["epochs"]]

fig, ax1 = plt.subplots(figsize=(8, 4.5))
ax1.plot(epochs_range, train_losses, 'b-o', label='Train Loss')
ax1.plot(epochs_range, val_losses, 'r-o', label='Val Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('BCE Loss', color='b')
ax1.legend(loc='upper left')
ax1.grid(True, linestyle='--', alpha=0.5)

ax2 = ax1.twinx()
ax2.plot(epochs_range, val_mrrs, 'g-s', label='Val MRR')
ax2.set_ylabel('Mean Reciprocal Rank (MRR)', color='g')
ax2.legend(loc='upper right')
plt.title('PatentRank Cross-Encoder Fine-Tuning Performance')
plt.tight_layout()
plt.show()

In [ ]:
# Step 10: Interactive Re-Ranking Demo
test_query = "semiconductor device scan test system boundary scan register chain"
candidates = [
    "A boundary scan bus error reporting circuitry loads an unused sentinel bit pattern...",
    "There is provided a scan test system comprising: a semiconductor device including a scan register connected between an input/output pin...",
    "A digital image compression processor employing discrete cosine transform algorithms..."
]

print(f"Test Query: '{test_query}'\n")
model.eval()
pairs = [[test_query, c] for c in candidates]
enc = tokenizer([p[0] for p in pairs], [p[1] for p in pairs], padding=True, truncation=True, return_tensors='pt').to(DEVICE)
with torch.no_grad():
    raw_scores = model(**enc).logits.squeeze(-1).cpu().tolist()

ranked = sorted(zip(candidates, raw_scores), key=lambda x: x[1], reverse=True)
for rank, (cand, sc) in enumerate(ranked, start=1):
    print(f"Rank #{rank} (Score: {sc:+.4f}): {cand[:85]}...")

In [ ]:
# Step 11: Package Trained Checkpoint for Local Inference
!zip -r patentrank_cross_encoder.zip models/patentrank-cross-encoder

print("Checkpoint zipped as 'patentrank_cross_encoder.zip'.")
try:
    from google.colab import files
    files.download('patentrank_cross_encoder.zip')
    print("Download initiated to your local machine!")
except Exception as e:
    print("To download manually, locate 'patentrank_cross_encoder.zip' in the Colab file tree.")